In [ ]:
# CELL 1 — Install dependencies
!pip install transformers datasets sentencepiece torch accelerate -q

In [ ]:
# CELL 2 — Imports
import json
import torch
from datasets import load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)


In [ ]:
# CELL 3 — Load Spider dataset (updated path)
dataset = load_dataset("xlangai/spider")
print(dataset)

In [ ]:
# CELL 4 — Load tokenizer and model

MODEL_NAME = "t5-base"  # Use t5-small if RAM is low in Colab
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

In [ ]:
# CELL 5 — Preprocessing: build input as "question + schema"
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 256

def preprocess(examples):
    # Build prompt: "translate to SQL: <question> | schema: <tables>"
    inputs = []
    for q, db in zip(examples["question"], examples["db_id"]):
        prompt = f"translate to SQL: {q} | database: {db}"
        inputs.append(prompt)

    targets = examples["query"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = dataset["train"].map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
tokenized_val   = dataset["validation"].map(preprocess, batched=True, remove_columns=dataset["validation"].column_names)

print("Train size:", len(tokenized_train))
print("Val size:  ", len(tokenized_val))

In [ ]:
# CELL 6 — Training arguments (fully updated)
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = Seq2SeqTrainingArguments(
    output_dir="./nl2sql_checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,   # ← removed tokenizer= argument
)

In [ ]:
# CELL 7 — Train! (takes ~1.5–2 hrs on Colab T4 GPU)
trainer.train()

In [ ]:
model.save_pretrained("./nl2sql_model")
tokenizer.save_pretrained("./nl2sql_model")
print("✅ Model saved!")


In [ ]:
# CELL 9 — Quick test (memory fixed)
import gc
torch.cuda.empty_cache()
gc.collect()

# Move model to CPU for inference (frees GPU)
model.to("cpu")
model.eval()

def predict_sql(question, db_name="orders"):
    prompt = f"translate to SQL: {question} | database: {db_name}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model.generate(inputs["input_ids"], max_length=256, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(predict_sql("show all customers"))
print(predict_sql("how many orders were placed last month"))
print(predict_sql("find the most expensive product"))

In [ ]:
# CELL 10 — Check Model Accuracy on Spider Validation Set
import gc
import torch
from tqdm import tqdm

torch.cuda.empty_cache()
gc.collect()
model.to("cpu")
model.eval()

def predict_sql(question, db_name="orders"):
    prompt = f"translate to SQL: {question} | database: {db_name}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model.generate(inputs["input_ids"], max_length=256, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Evaluation on first 200 validation examples ---
val_data = dataset["validation"]
N = 200  # increase to len(val_data) for full eval (slower)

exact_match = 0
partial_match = 0
results_log = []

for i in tqdm(range(N)):
    question  = val_data[i]["question"]
    db_name   = val_data[i]["db_id"]
    expected  = val_data[i]["query"].strip().lower()
    predicted = predict_sql(question, db_name).strip().lower()

    is_exact   = predicted == expected
    is_partial = (
        expected.split()[0] == predicted.split()[0]  # same SQL keyword (SELECT/INSERT etc)
        and any(w in predicted for w in expected.split() if len(w) > 3)
    )

    if is_exact:
        exact_match += 1
    if is_partial:
        partial_match += 1

    results_log.append({
        "question":  question,
        "expected":  expected,
        "predicted": predicted,
        "exact":     is_exact,
    })

print(f"\n{'='*50}")
print(f"  Evaluated on : {N} samples")
print(f"  Exact Match  : {exact_match}/{N} = {exact_match/N*100:.1f}%")
print(f"  Partial Match: {partial_match}/{N} = {partial_match/N*100:.1f}%")
print(f"{'='*50}")

# --- Show 10 sample predictions ---
print("\n📋 Sample Predictions:\n")
for r in results_log[:10]:
    status = "✅" if r["exact"] else "❌"
    print(f"{status} Q : {r['question']}")
    print(f"   Expected : {r['expected']}")
    print(f"   Got      : {r['predicted']}")
    print()

In [ ]:
!zip -r nl2sql_model.zip ./nl2sql_model
from google.colab import files
files.download("nl2sql_model.zip")